In [1]:


import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import resnet50
from sklearn.model_selection import train_test_split

from torch.ao.quantization import QConfig, QConfigMapping
from torch.ao.quantization.observer import MinMaxObserver, PerChannelMinMaxObserver
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

torch.backends.quantized.engine = "fbgemm"
DEVICE = torch.device("cpu")

# Dataset
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
NUM_CLASSES = len(dataset.classes)
print(f"Total images: {len(dataset)}, Classes: {NUM_CLASSES}")

targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for cid in np.unique(targets):
    idx = np.where(targets == cid)[0]
    tr, tmp = train_test_split(idx, test_size=0.15, random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

batch_size = 128
nw = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=nw, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True)
print("DataLoaders ready")

# Model + weights 
model_fp32 = resnet50(weights=None)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, NUM_CLASSES)

state = torch.load("resnet50_weights.pth", map_location="cpu")
if any(k.startswith("_orig_mod.") for k in state.keys()):
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
missing, unexpected = model_fp32.load_state_dict(state, strict=False)
print("Missing keys:", list(missing))
print("Unexpected keys:", list(unexpected))
model_fp32.eval().to(DEVICE)
print("FP32 ResNet-50 loaded")

#  per-channel symmetric qconfig 
per_channel_qconfig = QConfig(
    activation=MinMaxObserver.with_args(dtype=torch.quint8, qscheme=torch.per_tensor_affine),
    weight=PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric)
)
qconfig_mapping = QConfigMapping().set_global(per_channel_qconfig)

example_inputs = (torch.randn(1, 3, 224, 224),)
prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

# calibration
def calibrate_fx(m, loader, num_batches=20):
    m.eval()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            m(x.to("cpu"))
calibrate_fx(prepared, train_loader, num_batches=20)
print("Calibration done")

model_int8 = convert_fx(prepared).to("cpu").eval()
print("Converted to INT8 (per-channel symmetric)")

#  evaluation/size/runtime
def evaluate(model, loader):
    model.eval()
    corr, tot = 0, 0
    with torch.inference_mode():
        for x, y in loader:
            out = model(x.to("cpu"))
            pred = out.argmax(1).cpu()
            corr += (pred == y).sum().item()
            tot  += y.size(0)
    return 100.0 * corr / tot

def get_model_size(model, fname="__tmp__.pth"):
    torch.save(model.state_dict(), fname)
    sz = os.path.getsize(fname) / 1e6
    os.remove(fname)
    return sz

def benchmark(model, loader, num_batches=50):
    model.eval()
    t0 = time.time()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(x.to("cpu"))
    t1 = time.time()
    imgs = num_batches * loader.batch_size
    return (t1 - t0) / num_batches * 1000.0, (t1 - t0) / imgs * 1000.0

acc_fp32 = evaluate(model_fp32, test_loader)
acc_int8 = evaluate(model_int8, test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy (Per-channel Sym): {acc_int8:.2f}%")

fp32_sz = get_model_size(model_fp32)
int8_sz = get_model_size(model_int8)
print(f"FP32 size: {fp32_sz:.2f} MB | INT8 size: {int8_sz:.2f} MB")

b_ms_fp32, i_ms_fp32 = benchmark(model_fp32, test_loader)
b_ms_int8, i_ms_int8 = benchmark(model_int8, test_loader)
print(f"FP32 Inference: {b_ms_fp32:.2f} ms/batch | {i_ms_fp32:.2f} ms/image")
print(f"INT8 Inference: {b_ms_int8:.2f} ms/batch | {i_ms_int8:.2f} ms/image")



Total images: 100000, Classes: 200
DataLoaders ready
Missing keys: []
Unexpected keys: []
FP32 ResNet-50 loaded
Calibration done
Converted to INT8 (per-channel symmetric)
FP32 Accuracy: 73.48%
INT8 Accuracy (Per-channel Sym): 68.58%
FP32 size: 95.98 MB | INT8 size: 24.51 MB
FP32 Inference: 4024.28 ms/batch | 31.44 ms/image
INT8 Inference: 2795.85 ms/batch | 21.84 ms/image
